In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from pandas import DataFrame, merge, concat

from numpy import floor, ceil, cumsum, where
from collections import defaultdict
from logging import getLogger
import math, time


## Classes

from objects.Axle import Axle
from objects.Container import Container
from objects.ContainerSummary import ContainerSummary
from objects.ContainerLoadingRules import ContainerLoadingRules
from objects.Dimension import Dimension
from objects.Pallet import Pallet
from objects.Position import Position


In [3]:
logger = getLogger("load_planner")

# Assumptions
1. Units in the pallet are homogeneous
2. Units are calculated from item quantity to pallets
3. Dimensions are measured in inches (Later converted to Feet / other metrics)
4. **Routes have been pre-planned**
5. 

# Solver
1. Decide what items to load based on:
- Delivery date, priority, Item name, 
- Convert units into pallets
- Confirm the #of units shipped & update 
2. Grouping logic:
- Combine all items

### Optimizers to look into
1. 
## Feature enhancements
1. Stock pull-in from future (Early shipping)
2. Axle based handling-unit 📦(container) positioning
3. 

In [4]:
### Writing results to Database ###
# Write results to tables:
# 1. Handling_unit
# 2. Handling_unit_content
# 3. Handling_unit_position
# 4. Transport_equipment_assignment
# 5. route_planned


In [5]:
### Verify loaded 🚚 truck_equipment_assignment & handling_unit positions inside the container  

## Establish connection with Neon Database

In [4]:
### NeonDB Connection
import database.helper as db_helper
db_conn = db_helper.create_connection()

## Read Data From Database

In [5]:
item_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.item_master", connection=db_conn)
lane_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.lane_master", connection=db_conn)
load_equipment_metadata_df = db_helper.fetch_data(sql="select * from inventory_management.public.load_equipment_metadata", connection=db_conn)
location_df = db_helper.fetch_data(sql="select * from inventory_management.public.location", connection=db_conn)
shipment_plans_df = db_helper.fetch_data(sql="select * from inventory_management.public.shipment_plans", connection=db_conn)
sku_uom_df = db_helper.fetch_data(sql="select * from inventory_management.public.sku_unit_of_measure", connection=db_conn)
transport_asset_df = db_helper.fetch_data(sql="select * from inventory_management.public.transport_asset", connection=db_conn)


S:\git_repo\solutions-inventory-optimization\database\helper.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


## Create necessary features, calculations




In [6]:
def container_visualization(data:Container={}):
    from IPython.display import IFrame, display
    import json
    import urllib.parse
    encoded = ""
    if data:
        json_string = json.dumps(data)
        encoded = urllib.parse.quote(json_string)
    container_iframe = IFrame(
        src="http://localhost:5173/container-visualization" f"?data={encoded}",
        width="100%",
        height=450
    )

    display(container_iframe)

In [7]:
def mm_to_cm(series):
    """
    Convert millimeters to centimeters.

    Input:
        Pandas Series

    Returns:
        Pandas Series
    """

    return series.astype(float) / 10

In [8]:
def calculate_volume_m3(
    length_mm,
    width_mm,
    height_mm
):
    """
    Calculate cubic meters from
    millimeter dimensions.

    Accepts Pandas Series.
    """

    return (
        length_mm.astype(float)
        * width_mm.astype(float)
        * height_mm.astype(float)
    ) / 1_000_000_000

In [9]:
def mm3_to_m3(volume_mm3):

    return volume_mm3.astype(float) / 1_000_000_000

In [10]:
def calculate_pallet_floor_area_m2(
    length_mm,
    width_mm
):
    return (
        length_mm * width_mm
    ) / 1_000_000

In [ ]:
sku_uom_df = pd.concat(
    [
        sku_uom_df,
        sku_uom_df['pallet_dimensions'].apply(pd.Series)
    ],
    axis=1
)

sku_uom_column_mapper = {x:x for x in sku_uom_df.columns}
sku_uom_column_mapper['height_mm'] = 'pallet_height_mm'
sku_uom_column_mapper['width_mm'] = 'pallet_width_mm'
sku_uom_column_mapper['length_mm'] = 'pallet_length_mm'
sku_uom_df.rename(columns=sku_uom_column_mapper, inplace=True)

In [12]:


sku_uom_df['sku_id'] = sku_uom_df['sku_id'].astype(str)
shipment_plans_df['sku_id'] = shipment_plans_df['sku_id'].astype(str)

In [13]:
shipment_plans_df = pd.merge(
    left=shipment_plans_df, 
    right=sku_uom_df[['sku_id', 'unit_count_in_pallet', 'pallet_height_mm', 'pallet_width_mm', 'pallet_length_mm', ]], 
    left_on=['sku_id'], 
    right_on=['sku_id'], 
    how='left'
)


In [15]:

shipment_plans_df['planned_quantity_units_per_pallet'] = shipment_plans_df['planned_quantity'] / shipment_plans_df['unit_count_in_pallet']

shipment_plans_df['item_weight_kg'] = (shipment_plans_df['weight_kg']/shipment_plans_df['planned_quantity']).round(2)


# ----------------------------------
# Remaining Quantity
# ----------------------------------

shipment_plans_df["remaining_quantity"] = (
    shipment_plans_df["planned_quantity"]
    -
    shipment_plans_df["shipped_quantity"]
    .fillna(0)
)


In [16]:

# ----------------------------------
# Full Pallets
# ----------------------------------

shipment_plans_df["full_pallet_count"] = (
    shipment_plans_df["remaining_quantity"]
    //
    shipment_plans_df["unit_count_in_pallet"]
)

# ----------------------------------
# Remaining Units
# ----------------------------------

shipment_plans_df["remaining_units"] = (
    shipment_plans_df["remaining_quantity"]
    %
    shipment_plans_df["unit_count_in_pallet"]
)

# ----------------------------------
# Partial Pallet Fill %
# ----------------------------------

shipment_plans_df["pallet_fill_pct"] = np.where(
    shipment_plans_df["unit_count_in_pallet"] > 0,
    shipment_plans_df["remaining_units"]
    /
    shipment_plans_df["unit_count_in_pallet"],
    0
)


## Phase 1: 
Shipment Plans
<br>↓
Daily Shipment Selection
<br>↓
Build Full Pallets
<br>↓
Select Fixed 40FT Container
<br>↓
Create Container Object
<br>↓
Visualize

#### Future Scope:
- Multi-destination
- Route optimization
- Axle optimization
- Weight balancing
- Partial pallets
- Mixed pallets

In [ ]:
def build_daily_shipments(
	shipment_plans_df: pd.DataFrame,
	planning_date
) -> pd.DataFrame:
	"""
	Build shipment candidates for the optimizer.
	
	Assumptions
	----------
	- Single destination
	- Route already determined
	- Inventory exists
	- shipped_quantity = 0 for new planning runs
	- weight_kg represents total shipment weight
	for planned_quantity
	
	Future Enhancements
	-------------------
	- Inventory validation
	- Priority scoring
	- Multi-destination grouping
	- Deferred shipment tracking
	- Truck assignment
	
	Parameters
	----------
	shipment_plans_df : pd.DataFrame
	
	planning_date : str | datetime
	
	Returns
	-------
	shipment_candidate_df : pd.DataFrame
	"""

	df = shipment_plans_df.copy()

	# ----------------------------------
	# Standardize Dates
	# ----------------------------------

	df["estimated_delivery_date"] = pd.to_datetime(
		df["estimated_delivery_date"],
		errors="coerce"
	)

	planning_date = pd.to_datetime(
		planning_date
	)

	# ----------------------------------
	# Filter Due Shipments
	# ----------------------------------

	df = df[
		df["estimated_delivery_date"]
		<= planning_date
	]

	# ----------------------------------
	# Remove Invalid Quantities
	# ----------------------------------

	df = df[
		df["planned_quantity"] > 0
	]


	# ----------------------------------
	# Remove Fully Shipped Records
	# ----------------------------------

	df = df[
		df["remaining_quantity"] > 0
	]

	# ----------------------------------
	# Shipment Metrics
	# ----------------------------------

	df["shipment_volume_m3"] = (
		df["pallet_length_mm"]
		* df["pallet_width_mm"]
		* df["pallet_height_mm"]
		* df["planned_quantity_units_per_pallet"]
	) / 1_000_000_000

	df["shipment_floor_area_m2"] = (
		df["pallet_length_mm"]
		* df["pallet_width_mm"]
	) / 1_000_000

	# ----------------------------------
	# Priority Sort
	# ----------------------------------

	df = df.sort_values(
		by=[
			"priority",
			"estimated_delivery_date",
			"service_level"
		],
		ascending=[
			False,
			True,
			False
		]
	)

	# ----------------------------------
	# Reset Index
	# ----------------------------------

	df = df.reset_index(
		drop=True
	)

	return df


In [20]:
shipment_plans_df['estimated_delivery_date'].value_counts()

estimated_delivery_date
2026-05-04    1170
2026-05-11    1093
2026-04-27    1017
2026-04-20     945
2026-04-12     702
              ... 
2026-06-16       4
2026-08-02       2
2026-06-30       1
2026-07-21       1
2026-08-03       1
Name: count, Length: 114, dtype: int64

In [24]:
daily_shipments_df = build_daily_shipments(shipment_plans_df=shipment_plans_df, planning_date='2026-05-04')

In [ ]:
import numpy as np
import pandas as pd

def calculate_full_pallets(
    shipment_candidate_df: pd.DataFrame,
    allow_partial_pallet: bool = False,
    minimum_fill_pct: float = 1.0
) -> pd.DataFrame:
	"""
	Determine palletized shipment quantities.
	
	```
	Parameters
	----------
	shipment_candidate_df : pd.DataFrame
	
	allow_partial_pallet : bool
        Allow partially filled pallets.
	
	minimum_fill_pct : float
	
        Example:
            1.00 = 100%
            0.95 = 95%
            0.80 = 80%
	
	Returns
	-------
	shipment_load_df
	
	Additional Columns
	------------------
	full_pallet_count
	partial_pallet_count
	units_to_ship
	units_deferred
	pallet_fill_pct
	"""

	df = shipment_candidate_df.copy()

	# ----------------------------------
	# Partial Pallet Logic
	# ----------------------------------

	if allow_partial_pallet:

		df["partial_pallet_count"] = np.where(
			df["pallet_fill_pct"]
			>= minimum_fill_pct,
			1,
			0
		)

	else:
		df["partial_pallet_count"] = 0

	# ----------------------------------
	# Units To Ship
	# ----------------------------------

	df["units_to_ship"] = (
		df["full_pallet_count"]
		*
		df["unit_count_in_pallet"]
	)

	if allow_partial_pallet:

		df["units_to_ship"] += np.where(
			df["partial_pallet_count"] == 1,
			df["remaining_units"],
			0
		)

	# ----------------------------------
	# Deferred Units
	# ----------------------------------

	df["units_deferred"] = (
		df["remaining_quantity"]
		-
		df["units_to_ship"]
	)

	# ----------------------------------
	# Total Pallets To Build
	# ----------------------------------

	df["total_pallet_count"] = (
		df["full_pallet_count"]
		+
		df["partial_pallet_count"]
	)

	# ----------------------------------
	# Weight Allocation
	# ----------------------------------

	df["weight_per_unit_kg"] = (

		df["weight_kg"]
		/
		df["planned_quantity"]
	)

	df["weight_to_ship_kg"] = (
		df["weight_per_unit_kg"]
		*
		df["units_to_ship"]
	)

	df["weight_deferred_kg"] = (
		df["weight_per_unit_kg"]
		*
		df["units_deferred"]
	)

	return df



In [32]:
shipment_load_df = calculate_full_pallets(shipment_candidate_df=daily_shipments_df)

In [37]:
import uuid
import pandas as pd


def build_pallets(
	shipment_load_df: pd.DataFrame
) -> pd.DataFrame:
	"""
	Convert shipment demand into physical pallets.
	
	Assumptions
	-----------
	- 1 pallet = 1 SKU
	- 1 pallet = 1 destination
	- No stacking
	- No position assignment
	
	Future
	------
	- Mixed SKU pallets
	- Mixed destination pallets
	- Stackable pallets
	- Dynamic pallet sizing
	
	Returns
	-------
	pallet_df
	"""

	pallet_records = []

	shipment_load_df["total_pallet_count"] = (
		shipment_load_df["total_pallet_count"]
		.fillna(0)
		.astype(int)
	)

	shipment_load_df["unit_count_in_pallet"] = (
		shipment_load_df["unit_count_in_pallet"]
		.fillna(0)
		.astype(int)
	)


	for _, row in shipment_load_df.iterrows():

		pallet_count = int(
			pd.to_numeric(
				row["total_pallet_count"],
				errors="coerce"
			)
			if pd.notna(
				row["total_pallet_count"]
			)
			else 0
		)

		if pallet_count <= 0:
			continue

		units_per_pallet = int(
			row["unit_count_in_pallet"]
		)

		weight_per_unit = (
			row["weight_kg"]
			/
			row["planned_quantity"]
		)

		pallet_weight = (
			weight_per_unit
			*
			units_per_pallet
		)

		pallet_volume_m3 = (
			row["pallet_length_mm"]
			*
			row["pallet_width_mm"]
			*
			row["pallet_height_mm"]
		) / 1_000_000_000

		pallet_floor_area_m2 = (
			row["pallet_length_mm"]
			*
			row["pallet_width_mm"]
		) / 1_000_000

		for pallet_sequence in range(
			1,
			pallet_count + 1
		):

			pallet_records.append({

				"pallet_id":

					f"PALLET_{uuid.uuid4().hex[:12].upper()}",

				"shipment_id":

					row["shipment_id"],

				"sku_id":

					row["sku_id"],

				"origin_location_id":

					row["origin_location_id"],

				"destination_location_id":

					row["destination_location_id"],

				"pallet_sequence":

					pallet_sequence,

				"units_in_pallet":

					units_per_pallet,

				"weight_kg":

					pallet_weight,

				"length_mm":

					row["pallet_length_mm"],

				"width_mm":

					row["pallet_width_mm"],

				"height_mm":

					row["pallet_height_mm"],

				"volume_m3":

					pallet_volume_m3,

				"floor_area_m2":

					pallet_floor_area_m2,

				"priority":

					row["priority"],

				"load_order":

					0,

				"unload_order":

					0,

				"position_x":

					0,

				"position_y":

					0,

				"position_z":

					0
			})

	pallet_df = pd.DataFrame(
		pallet_records
	)

	return pallet_df


In [38]:
pallets_df = build_pallets(shipment_load_df=shipment_load_df)

In [40]:
load_equipment_metadata_df.columns.to_list()

['equipment_id',
 'equipment_name',
 'equipment_type',
 'length_mm',
 'width_mm',
 'height_mm',
 'internal_length_mm',
 'internal_width_mm',
 'internal_height_mm',
 'max_payload_weight_kg',
 'tare_weight_kg',
 'door_width_mm',
 'door_height_mm',
 'refrigeration_capable',
 'temperature_min_c',
 'temperature_max_c',
 'max_stack_height_mm',
 'axle_configuration',
 'created_at']

In [44]:
def build_fixed_container(
	load_equipment_metadata_df
):
	"""
	Build fixed 40FT container.
	
	Returns
	-------
	Container
	"""

	equipment_row = (

		load_equipment_metadata_df

		.loc[
			load_equipment_metadata_df[
				"equipment_name"
			]

			.str.upper()

			.str.contains(
				"40FT",
				na=False
			)
		]

		.iloc[0]
	)

	container = Container(

		containerId=str(
			equipment_row["equipment_id"]
		),

		containerType=
			equipment_row["equipment_name"],

		length=
			equipment_row["length_mm"],

		width=
			equipment_row["width_mm"],

		height=
			equipment_row["height_mm"],

		internal_length=
			equipment_row["internal_length_mm"],

		internal_width=
			equipment_row["internal_width_mm"],

		internal_height=
			equipment_row["internal_height_mm"],

		maxPayloadWeight=
			equipment_row["max_payload_weight_kg"],

		tareWeight=
			equipment_row["tare_weight_kg"],

		maxVolume=(

			equipment_row["internal_length_mm"]
			*
			equipment_row["internal_width_mm"]
			*
			equipment_row["internal_height_mm"]

		) / 1_000_000_000,

		door_width=
			equipment_row["door_width_mm"],

		door_height=
			equipment_row["door_height_mm"],

		pallets=[]
	)

	return container


In [45]:
container = build_fixed_container(load_equipment_metadata_df=load_equipment_metadata_df)

In [48]:
def validate_container_capacity(
	container,
	pallet_df
):
	"""
	Validate whether pallet load fits
	inside the container.
	
	Returns
	-------
	dict
	"""

	total_weight_kg = (
		pallet_df["weight_kg"]
		.sum()
	)

	total_floor_area_m2 = (
		pallet_df["floor_area_m2"]
		.sum()
	)

	total_volume_m3 = (
		pallet_df["volume_m3"]
		.sum()
	)

	container_floor_area_m2 = (
		container.internal_length
		*
		container.internal_width
	) / 1_000_000

	container_volume_m3 = (
		container.internal_length
		*
		container.internal_width
		*
		container.internal_height
	) / 1_000_000_000

	weight_utilization_pct = (
		total_weight_kg
		/
		container.maxPayloadWeight
		* 100
	)

	floor_utilization_pct = (
		total_floor_area_m2
		/
		container_floor_area_m2
		* 100
	)

	volume_utilization_pct = (
		total_volume_m3
		/
		container_volume_m3
		* 100
	)

	fits = (
		total_weight_kg
		<= container.maxPayloadWeight
		and
		total_floor_area_m2
		<= container_floor_area_m2
	)

	return {

		"fits": fits,

		"total_weight_kg":
			round(
				total_weight_kg,
				2
			),

		"total_floor_area_m2":
			round(
				total_floor_area_m2,
				2
			),

		"total_volume_m3":
			round(
				total_volume_m3,
				2
			),

		"weight_utilization_pct":
			round(
				weight_utilization_pct,
				2
			),

		"floor_utilization_pct":
			round(
				floor_utilization_pct,
				2
			),

		"volume_utilization_pct":
			round(
				volume_utilization_pct,
				2
			)
	}


In [50]:
validate_container_capacity(container=container, pallet_df=pallets_df)

{'fits': np.False_,
 'total_weight_kg': np.float64(109062249855.0),
 'total_floor_area_m2': np.float64(256191.74),
 'total_volume_m3': np.float64(346706.84),
 'weight_utilization_pct': np.float64(436248999.42),
 'floor_utilization_pct': np.float64(905369.95),
 'volume_utilization_pct': np.float64(512012.62)}

In [52]:
pallets_df.columns.tolist()

['pallet_id',
 'shipment_id',
 'sku_id',
 'origin_location_id',
 'destination_location_id',
 'pallet_sequence',
 'units_in_pallet',
 'weight_kg',
 'length_mm',
 'width_mm',
 'height_mm',
 'volume_m3',
 'floor_area_m2',
 'priority',
 'load_order',
 'unload_order',
 'position_x',
 'position_y',
 'position_z']

In [ ]:
def place_pallets(
    pallet_df,
    container
):
    """
    Simple floor placement algorithm.

    No stacking.
    Width-first placement.

    Returns
    -------
    pallet_position_df
    """

    df = pallet_df.copy()

    current_x = 0
    current_z = 0

    row_depth = 0

    container_length = (
        container.internal_length
    )

    container_width = (
        container.internal_width
    )

    for idx in df.index:

        pallet_length = (
            df.loc[idx, "length_mm"]
        )

        pallet_width = (
            df.loc[idx, "width_mm"]
        )

        # ----------------------
        # New Row
        # ----------------------

        if (
            current_z
            + pallet_width
            >
            container_width
        ):

            current_x += row_depth

            current_z = 0

            row_depth = 0

        # ----------------------
        # Capacity Check
        # ----------------------

        if (
            current_x
            + pallet_length
            >
            container_length
        ):

            raise ValueError(
                "Container capacity exceeded."
            )

        # ----------------------
        # Assign Position
        # ----------------------

        df.loc[idx, "position_x"] = (
            current_x
        )

        df.loc[idx, "position_y"] = 0

        df.loc[idx, "position_z"] = (
            current_z
        )

        # ----------------------
        # Advance
        # ----------------------

        current_z += pallet_width

        row_depth = max(
            row_depth,
            pallet_length
        )

    return df

In [ ]:
# Step 6
def validate_capacity():
    """
    Purpose:

    Check whether pallets fit.

    Checks

    Weight
    total_pallet_weight

    vs

    container.maxPayloadWeight
    Floor Area
    Σ pallet_area

    vs

    container_floor_area
    Volume
    Σ pallet_volume

    vs

    container_volume

    Output

    {
        fits: True,
        reason: ""
    }
    Future

    Multi-container allocation."""

In [ ]:
def calculate_center_of_gravity(
    pallet_df
):
    """
    Calculate load center of gravity.

    Returns
    -------
    dict
    """

    total_weight = (

        pallet_df["weight_kg"]

        .sum()
    )

    if total_weight == 0:

        return {

            "cg_x": 0,

            "cg_y": 0,

            "cg_z": 0
        }

    cg_x = (

        pallet_df["weight_kg"]

        *

        pallet_df["position_x"]

    ).sum() / total_weight

    cg_z = (

        pallet_df["weight_kg"]

        *

        pallet_df["position_z"]

    ).sum() / total_weight

    return {

        "cg_x": round(
            cg_x,
            2
        ),

        "cg_y": 0,

        "cg_z": round(
            cg_z,
            2
        )
    }

In [ ]:
def calculate_weight_distribution(
    pallet_df,
    container
):

    center_x = (
        container.internal_length
        / 2
    )

    center_z = (
        container.internal_width
        / 2
    )

    front_weight = (

        pallet_df[
            pallet_df["position_x"]
            < center_x
        ]

        ["weight_kg"]

        .sum()
    )

    rear_weight = (

        pallet_df[
            pallet_df["position_x"]
            >= center_x
        ]

        ["weight_kg"]

        .sum()
    )

    left_weight = (

        pallet_df[
            pallet_df["position_z"]
            < center_z
        ]

        ["weight_kg"]

        .sum()
    )

    right_weight = (

        pallet_df[
            pallet_df["position_z"]
            >= center_z
        ]

        ["weight_kg"]

        .sum()
    )

    return {

        "front_weight_kg":
            round(front_weight, 2),

        "rear_weight_kg":
            round(rear_weight, 2),

        "left_weight_kg":
            round(left_weight, 2),

        "right_weight_kg":
            round(right_weight, 2)
    }

In [ ]:
def optimize_pallet_sequence(
    pallet_df
):
    """
    Sort pallets before placement.

    Returns
    -------
    pallet_df
    """

    df = pallet_df.copy()

    df = (

        df

        .sort_values(

            by=[
                "priority",
                "weight_kg"
            ],

            ascending=[
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )

    df["load_order"] = (

        df.index + 1
    )

    df["unload_order"] = (

        df.index + 1
    )

    return df

In [ ]:
# Step 7
def build_container_summary():
    """
    Purpose:

    Populate:

    ContainerSummary

    Output

    ContainerSummary(
        shipmentId=...
    )
    Future

    Multiple shipments."""

In [ ]:
# Step 8
def build_container():
    """
    Purpose:

    Combine everything.

    Output

    Container(
        pallets=[
            ...
        ]
    )

    Pallet positions:

    Position(
        x=0,
        y=0,
        z=0
    )

    temporary."""

# Phase 2 (Next)

Only after Phase 1 is working:

Container
<br>      ↓
Pallet Sorting
<br>      ↓
Pallet Placement
<br>      ↓
Weight Distribution
<br>      ↓
CG Calculation
<br>      ↓
Axle Load Calculation
<br>      ↓
Visualization Update

This is where the actual loading optimizer begins.

# Older version of phase 1
# IGNORE

In [ ]:
# Phase 1A - Build Shipment candidates

def build_shipment_candidates(
    shipment_plans_df,
    planning_date
):
    """

    Output: candidate_shipments_df

    Assumptions:
    1. All inventory exists
    2. All shipments are valid
    
    Future Optimizations:
    1. Customer Priority
    2. Revenue Priority
    3. Margin Priority
    4. Late Shipment Priority
    """
    

In [ ]:
# Phase 1B - Inventory Validation

def validate_inventory(
    candidate_shipments_df,
    inventory_df
):
    """
    Determine:
    1. Can ship
    2. Cannot ship
    3. Partial ship
    
    Output: validated_shipments_df

    Assumptions:
    1. Single Warehouse
    
    Future Optimizations:
    1. Multi-location sourcing
    """
    

In [ ]:
# Phase 1C - Shipment Consolidation

def consolidate_shipment(
    validated_shipments_df,
):
    """
    Group by:
        1. origin_location_id
        2. destination_location_id
        3. sku_id

    Output: shipment_summary_df

    Example:
        5 orders → 1 shipment aggregate

    Why?
        Pallet Building becomes easier
    
    Future Optimizations:
        1. Maintain order lineage
    """
    

In [ ]:
# Phase 1D - Container Feasibility Analysis

def calculate_shipping_requirements(
    shipment_summary_df,
    item_master_df,
    sku_uom_df
):
    """
    This is something SupplyPlan doesn't do because it focuses on capacities.
    You need it before building pallets.

    Output:
        shipping_requirements

    """
    

In [ ]:
# Phase 1E - Equipment Feasibility

def find_feasible_containers(
    shipping_requirements,
    load_equipment_metadata_df
):
    """
    Checks based on Volume, weight, floor area, 
    
    Example:
        53FT Dry Van
        PASS

        40FT Reefer
        FAIL

        20FT Container
        FAIL

    """
    

In [ ]:
# Phase 1F - Select Container

def select_container(
    feasible_containers_df
):
    """
    Current:
        Smallest feasible container
    Future Optimization:
    1. Cost
    2. Distance
    3. Carbon
    4. Asset Availability
    """
    

In [ ]:
# Phase 1G - Truck Build Day

def build_daily_truck_load(
    selected_container,
    shipment_summary_df
):
    """
    
    Output:
        truck_build_df
    """
    

### Phase 2:
Truck Build
<br>    ↓
Pallet Formation
<br>    ↓
Pallet Grouping
<br>    ↓
Unload Sequence
<br>    ↓
Container Placement

In [43]:
container_visualization()

In [ ]:

class LoadPlanPreprocessor:
	"""
	Validates, cleans, and reshapes all inputs into the formats the solver needs.
	Called once before the day loop starts.
	"""

	def __init__(self, engine):
		self.engine = engine
		self.c = engine.c # constants shorthand

	# ── 2a. Parameter validation ─────────────────────────────────────────────
	
	def check_inputs(self):
		"""Exit early if mandatory tables are missing or empty."""
		cfg = self.engine
		c = self.c
		missing = []

		if not isinstance(cfg.shipment_plans_df, DataFrame) or cfg.shipment_plans_df.empty:
			missing.append("shipment_plans_df")
		if not isinstance(cfg.item_master_df, DataFrame) or cfg.item_master_df.empty:
			missing.append("item_master_df")
		if not isinstance(cfg.load_equipment_metadata_df, DataFrame) or cfg.load_equipment_metadata_df.empty:
			missing.append("load_equipment_metadata_df")
		if not isinstance(cfg.lane_master_df, DataFrame) or cfg.lane_master_df.empty:
			missing.append("lane_master_df")

		if missing:
			cfg.no_input = True
			logger.error(f"LoadPlanner: missing required inputs: {missing}")

	# ── 2b. Item master ───────────────────────────────────────────────────────

	def preprocess_item_master(self):
		"""
		Fill nulls with safe defaults, derive floor_area and volume if not present,
		build a dict keyed by item_id for O(1) lookup in the solver.
		"""
		cfg = self.engine
		c = self.c
		df = cfg.item_master_df.copy()

		# TODO: adjust column names to match your actual item_master_df schema

		df[c.m_item_weight_per_uom].fillna(0, inplace=True)
		df[c.m_item_length].fillna(0, inplace=True)
		df[c.m_item_width].fillna(0, inplace=True)
		df[c.m_item_height].fillna(0, inplace=True)
		df[c.m_stackable].fillna(True, inplace=True)
		df[c.m_fragile].fillna(False, inplace=True)
		df[c.m_units_per_pallet].fillna(1, inplace=True)

		# derive floor area and volume if not explicit
		if c.m_item_floor_area not in df.columns:
			df[c.m_item_floor_area] = df[c.m_item_length] * df[c.m_item_width]
		if c.m_item_volume not in df.columns:
			df[c.m_item_volume] = df[c.m_item_floor_area] * df[c.m_item_height]

		# dict: item_id → {weight_kg, floor_area_m2, volume_m3, stackable, ...}
		cfg.item_lookup = df.set_index(c.ITEM).to_dict(orient="index")
		cfg.item_master_df = df

	# ── 2c. Equipment metadata ────────────────────────────────────────────────

	def preprocess_equipment(self):
		"""
		Fill missing capacities, build per-truck lookup dict,
		merge with transport_asset_df to get day-level availability.
		"""
		cfg = self.engine
		c = self.c
		df = cfg.load_equipment_metadata_df.copy()

		# TODO: adjust column names to match your actual equipment schema

		df[c.m_truck_max_payload_kg].fillna(float("inf"), inplace=True)
		df[c.m_truck_volume_m3].fillna(float("inf"), inplace=True)
		df[c.m_truck_floor_area_m2].fillna(float("inf"), inplace=True)
		df[c.m_truck_max_front_axle_kg].fillna(float("inf"), inplace=True)
		df[c.m_truck_max_rear_axle_kg].fillna(float("inf"), inplace=True)
		df[c.m_truck_tare_weight_kg].fillna(0, inplace=True)

		# derive floor area if columns exist
		if c.m_truck_floor_area_m2 not in df.columns:
			df[c.m_truck_floor_area_m2] = (
				df[c.m_truck_internal_length_m] * df[c.m_truck_internal_width_m]
			)

		# merge availability
		if isinstance(cfg.transport_asset_df, DataFrame) and not cfg.transport_asset_df.empty:
			df = merge(df, cfg.transport_asset_df,
				on=[c.TRUCK], how="left")
			df[c.m_asset_available].fillna(1, inplace=True)
		else:
			df[c.m_asset_available] = 1

		cfg.equipment_df = df
		cfg.equipment_lookup = df.set_index(c.TRUCK).to_dict(orient="index")

	# ── 2d. Lane master ───────────────────────────────────────────────────────

	def preprocess_lanes(self):
		"""
		Build a dict: truck_id → ordered list of (stop_order, location_id).
		This drives the LIFO loading order (last stop loads first).
		"""
		cfg = self.engine
		c = self.c
		df = cfg.lane_master_df.sort_values([c.TRUCK, c.STOP_ORDER])

		cfg.lane_lookup = (
			df.groupby(c.TRUCK)
				.apply(lambda g: list(zip(g[c.STOP_ORDER], g[c.LOCATION])))
				.to_dict()
		)

	# ── 2e. Shipment plan pivot ───────────────────────────────────────────────

	def preprocess_shipment_plan(self):
		"""
		Pivot shipment_plans_df so days become columns — same pattern as TLB.
		Also merge item master columns needed during loading.
		"""
		cfg = self.engine
		c = self.c
		df = cfg.shipment_plans_df.copy()

		# derive pallet count from qty
		df = merge(df,
			cfg.item_master_df[[c.ITEM, c.m_units_per_pallet,
				c.m_item_weight_per_uom,
				c.m_item_floor_area, c.m_item_volume,
				c.m_stackable, c.m_fragile]],
			on=c.ITEM, how="left")

		df[c.m_pallets_needed] = ceil(
			df[c.m_demand_qty] / df[c.m_units_per_pallet]
		)
		df[c.m_pallet_weight_kg] = (
			df[c.m_units_per_pallet] * df[c.m_item_weight_per_uom]
		)
		df[c.m_pallet_floor_area] = df[c.m_item_floor_area]
		df[c.m_pallet_volume] = df[c.m_item_volume] * df[c.m_units_per_pallet]

		# pivot: rows = (truck, item, location), cols = day columns
		pivot_dims = [c.TRUCK, c.LANE, c.ITEM, c.LOCATION,
			c.m_pallet_weight_kg, c.m_pallet_floor_area,
			c.m_pallet_volume, c.m_stackable, c.m_fragile]

		cfg.days = sorted(df[c.DAY].unique().tolist())
		pivot = df.pivot_table(
			values=c.m_pallets_needed,
			index=pivot_dims,
			columns=c.DAY,
			aggfunc="sum",
			fill_value=0,
		).reset_index()

		# add priority columns
		if c.m_demand_priority in df.columns:
			prio = df.pivot_table(
				values=c.m_demand_priority,
				index=[c.TRUCK, c.ITEM, c.LOCATION],
				columns=c.DAY,
				aggfunc="min",
			).reset_index()
			prio.columns = [
				f"{col}_priority" if col in cfg.days else col
				for col in prio.columns
			]
			pivot = merge(pivot, prio, on=[c.TRUCK, c.ITEM, c.LOCATION], how="left")

		# fill missing priority with inf
		for day in cfg.days:
			col = f"{day}_priority"
			if col not in pivot.columns:
				pivot[col] = float("inf")
		pivot.fillna({f"{d}_priority": float("inf") for d in cfg.days}, inplace=True)

		cfg.shipment_pivot = pivot

	# ── 2f. Run all preprocessors ─────────────────────────────────────────────

	def run(self):
		self.check_inputs()
		if self.engine.no_input:
			return
		self.preprocess_item_master()
		self.preprocess_equipment()
		self.preprocess_lanes()
		self.preprocess_shipment_plan()
		logger.info("LoadPlanner: preprocessing complete")



### ─────────────────────────────────────────────────────────────────────────────
3. CONSTRAINT CHECKER
### ─────────────────────────────────────────────────────────────────────────────

In [23]:

class ConstraintChecker:
	"""
	Pure functions that test whether adding a pallet to a truck is feasible.
	No state — called inline during loading.
	"""

	def __init__(self, engine):
		self.engine = engine
		self.c = engine.c

	def remaining_weight(self, truck_state: dict) -> float:
		"""Payload headroom remaining (kg)."""
		c = self.c
		return (
			truck_state[c.m_truck_max_payload_kg]
			- truck_state[c.m_load_weight_kg]
		)

	def remaining_floor_area(self, truck_state: dict) -> float:
		"""Floor area headroom remaining (m²)."""
		c = self.c
		return (
			truck_state[c.m_truck_floor_area_m2]
			- truck_state[c.m_cumulative_floor_area]
		)

	def can_load_pallet(self, pallet: dict, truck_state: dict) -> bool:
		"""
		Returns True only if all hard constraints are satisfied.
		Extend this function as your rules grow.
		"""
		c = self.c

		# weight
		if pallet[c.m_pallet_weight_kg] > self.remaining_weight(truck_state):
			return False

		# floor area (stackable pallets can share floor space — TODO: refine)
		if not pallet.get(c.m_stackable, True):
			if pallet[c.m_pallet_floor_area] > self.remaining_floor_area(truck_state):
				return False

		# fragile items must be loaded LAST (top layer)
		# TODO: implement layer-aware fragility check using POSITION_Z

		return True

	def axle_loads_after_adding(self, pallet: dict, truck_state: dict,
		position_x: float) -> tuple:
		"""
		Estimate front and rear axle loads after placing a pallet at position_x.
		Uses simple lever-arm (beam equation) around the rear axle.
		
		Returns (front_axle_kg, rear_axle_kg).
		"""
		c = self.c
		eq = self.engine.equipment_lookup.get(truck_state[c.TRUCK], {})

		rear_axle_x = eq.get(c.m_truck_rear_axle_x, truck_state.get(c.m_truck_rear_axle_x, 5.0))
		front_axle_x = eq.get(c.m_truck_front_axle_x, truck_state.get(c.m_truck_front_axle_x, 1.5))
		axle_base = rear_axle_x - front_axle_x

		pallet_weight = pallet[c.m_pallet_weight_kg]
		# distance of pallet from rear axle (positive = ahead of rear axle)
		d_from_rear = rear_axle_x - position_x

		if axle_base <= 0:
			# fallback: split evenly
			delta_front = delta_rear = pallet_weight / 2
		else:
			# proportion on front axle = d_from_rear / axle_base
			delta_front = pallet_weight * (d_from_rear / axle_base)
			delta_rear = pallet_weight - delta_front

		new_front = truck_state.get(c.m_axle_front_load_kg, 0) + delta_front
		new_rear = truck_state.get(c.m_axle_rear_load_kg, 0) + delta_rear
		return new_front, new_rear

	def axle_compliant(self, truck_id: str, front_kg: float, rear_kg: float) -> bool:
		"""Check both axles are within rated limits."""
		c = self.c
		eq = self.engine.equipment_lookup.get(truck_id, {})
		return (
			front_kg <= eq.get(c.m_truck_max_front_axle_kg, float("inf"))
			and rear_kg <= eq.get(c.m_truck_max_rear_axle_kg, float("inf"))
		)



#### ─────────────────────────────────────────────────────────────────────────────
4. POSITION ENGINE
#### ─────────────────────────────────────────────────────────────────────────────


In [24]:
class PositionEngine:
	"""
	Assigns (x, y, z) coordinates to each pallet inside the truck container.
	
	Coordinate system:
	  x — longitudinal, 0 = front cab wall, increases toward rear doors
	  y — lateral, 0 = left wall, increases rightward
	  z — vertical, 0 = floor, increases upward
	
	Loading strategy: rear-to-front (last delivery stop loads first → LIFO).
	"""

	def __init__(self, engine):
		self.engine = engine
		self.c = engine.c

	def assign_position(self, pallet: dict, truck_state: dict,
		stop_order: int) -> dict:
		"""
		Given the current truck state (list of loaded pallets, remaining space),
		return updated pallet dict with POSITION_X, POSITION_Y, POSITION_Z set.
		
		Placement logic:
		  1. Sort stops so the LAST stop loads at the rear (high X, near doors).
		  2. Within the same stop, pack left-to-right, then stack if stackable.
		  3. Fragile items always land on top (highest Z in their column).
		
		TODO: replace this naive sequential placer with a bin-packing algorithm
		      or a grid-slot system for production use.
		"""
		c = self.c
		eq = self.engine.equipment_lookup.get(truck_state.get(c.TRUCK, ""), {})

		truck_length = eq.get(c.m_truck_internal_length_m, 13.6)
		truck_width = eq.get(c.m_truck_internal_width_m, 2.4)

		# Naive: assign x based on stop_order (last stop = highest x = rear)
		max_stop = truck_state.get("max_stop_order", 1)
		x_zone_length = truck_length / max(max_stop, 1)
		pos_x = (stop_order - 1) * x_zone_length # earlier stop = closer to cab

		# y: pack left to right within the zone
		# TODO: track used y within each x-zone, implement 2-D bin packing
		pos_y = truck_state.get("current_y_offset", 0)

		# z: stack if stackable and previous pallet is also stackable
		# TODO: implement column-aware stacking with height limits
		pos_z = 0

		pallet[c.POSITION_X] = round(pos_x, 3)
		pallet[c.POSITION_Y] = round(pos_y, 3)
		pallet[c.POSITION_Z] = round(pos_z, 3)
		return pallet

	def compute_centre_of_gravity(self, loaded_pallets: list) -> tuple:
		"""
		Returns (cog_x, cog_y) — centre of gravity of the load.
		Used for axle balance reporting.
		"""
		c = self.c
		total_weight = sum(p[c.m_pallet_weight_kg] for p in loaded_pallets)
		if total_weight == 0:
			return 0.0, 0.0
		cog_x = sum(p[c.m_pallet_weight_kg] * p[c.POSITION_X] for p in loaded_pallets) / total_weight
		cog_y = sum(p[c.m_pallet_weight_kg] * p[c.POSITION_Y] for p in loaded_pallets) / total_weight
		return round(cog_x, 3), round(cog_y, 3)



#### ─────────────────────────────────────────────────────────────────────────────
5. PALLET LOADER
#### ─────────────────────────────────────────────────────────────────────────────


In [25]:

class PalletLoader:
	"""
	Selects which pallets go on which truck for a given day,
	respecting constraints and maximising utilisation.
	
	Mirrors BulkLoader from TLB:
	  - sort pallets by stop order (LIFO) then priority then weight
	  - iterate trucks, fill each in turn
	  - overflow → push-out to next day
	  - empty space → pull-in from next day (optional)
	"""

	def __init__(self, engine):
		self.engine = engine
		self.c = engine.c
		self.checker = ConstraintChecker(engine)
		self.positioner = PositionEngine(engine)

	# ── helpers ───────────────────────────────────────────────────────────────

	def _init_truck_state(self, truck_id: str, day: str) -> dict:
		"""Create a fresh mutable state dict for one truck on one day."""
		c = self.c
		eq = self.engine.equipment_lookup.get(truck_id, {})
		stops = self.engine.lane_lookup.get(truck_id, [])
		return {
			c.TRUCK: truck_id,
			c.DAY: day,
			c.m_truck_max_payload_kg: eq.get(c.m_truck_max_payload_kg, float("inf")),
			c.m_truck_floor_area_m2: eq.get(c.m_truck_floor_area_m2, float("inf")),
			c.m_truck_volume_m3: eq.get(c.m_truck_volume_m3, float("inf")),
			c.m_truck_max_front_axle_kg: eq.get(c.m_truck_max_front_axle_kg, float("inf")),
			c.m_truck_max_rear_axle_kg: eq.get(c.m_truck_max_rear_axle_kg, float("inf")),
			c.m_truck_front_axle_x: eq.get(c.m_truck_front_axle_x, 1.5),
			c.m_truck_rear_axle_x: eq.get(c.m_truck_rear_axle_x, 5.0),
			c.m_load_weight_kg: 0.0,
			c.m_cumulative_floor_area: 0.0,
			c.m_axle_front_load_kg: eq.get(c.m_truck_tare_weight_kg, 0) * 0.5,
			c.m_axle_rear_load_kg: eq.get(c.m_truck_tare_weight_kg, 0) * 0.5,
			"max_stop_order": max((s for s, _ in stops), default=1),
			"loaded_pallets": [],
			"current_y_offset": 0.0,
			"load_sequence": 0,
		}

	def _get_stop_order(self, truck_id: str, location_id: str) -> int:
		"""Look up where on the route this location sits."""
		stops = self.engine.lane_lookup.get(truck_id, [])
		for order, loc in stops:
			if loc == location_id:
				return order
		return 999 # unknown stop — load last (near cab)

	# ── main loading function ──────────────────────────────────────────────────

	def load_truck_for_day(self, truck_id: str, day: str,
		demand_rows: DataFrame) -> tuple:
		"""
		Greedy sequential loader for one truck on one day.
		Returns (loaded_list, overflow_df).
		
		loaded_list  — list of dicts, one per pallet loaded
		overflow_df  — rows from demand_rows that did not fit
		"""
		c = self.c
		truck_state = self._init_truck_state(truck_id, day)
		loaded = []
		overflow_idx = []

		# Sort: last delivery stop → rear of truck → loads first (LIFO)
		demand_rows = demand_rows.copy()
		demand_rows["_stop_order"] = demand_rows[c.LOCATION].apply(
			lambda loc: self._get_stop_order(truck_id, loc)
		)
		priority_col = f"{day}_priority"
		demand_rows.sort_values(
			["_stop_order", priority_col, c.m_pallet_weight_kg],
			ascending=[False, True, False], # last stop first, highest priority first
			inplace=True,
		)
		demand_rows.reset_index(drop=True, inplace=True)

		for idx, row in demand_rows.iterrows():
			pallets_to_load = int(floor(row[day]))
			if pallets_to_load <= 0:
				continue

			stop_order = int(row["_stop_order"])
			pallet_template = {
				c.ITEM: row[c.ITEM],
				c.LOCATION: row[c.LOCATION],
				c.STOP_ORDER: stop_order,
				c.LANE: row.get(c.LANE, ""),
				c.m_pallet_weight_kg: row[c.m_pallet_weight_kg],
				c.m_pallet_floor_area: row[c.m_pallet_floor_area],
				c.m_pallet_volume: row[c.m_pallet_volume],
				c.m_stackable: row.get(c.m_stackable, True),
				c.m_fragile: row.get(c.m_fragile, False),
			}

			loaded_count = 0
			for _ in range(pallets_to_load):
				pallet = pallet_template.copy()

				if not self.checker.can_load_pallet(pallet, truck_state):
					break # remaining pallets for this item also won't fit

				# assign position
				pos_x = (stop_order - 1) * (
					self.engine.equipment_lookup.get(truck_id, {})
						.get(c.m_truck_internal_length_m, 13.6)
					/ max(truck_state["max_stop_order"], 1)
				)
				pallet = self.positioner.assign_position(pallet, truck_state, stop_order)

				# axle check
				new_front, new_rear = self.checker.axle_loads_after_adding(
					pallet, truck_state, pallet[c.POSITION_X]
				)
				# Note: axle check is advisory only here; set to hard-fail if needed
				# if not self.checker.axle_compliant(truck_id, new_front, new_rear):
				#     break

				# commit pallet to truck
				truck_state["load_sequence"] += 1
				pallet[c.SEQUENCE] = truck_state["load_sequence"]
				pallet[c.TRUCK] = truck_id
				pallet[c.DAY] = day
				pallet[c.m_axle_front_load_kg] = new_front
				pallet[c.m_axle_rear_load_kg] = new_rear

				truck_state[c.m_load_weight_kg] += pallet[c.m_pallet_weight_kg]
				truck_state[c.m_cumulative_floor_area] += pallet[c.m_pallet_floor_area]
				truck_state[c.m_axle_front_load_kg] = new_front
				truck_state[c.m_axle_rear_load_kg] = new_rear
				truck_state["loaded_pallets"].append(pallet)

				loaded.append(pallet)
				loaded_count += 1

			# record overflow
			overflow_qty = pallets_to_load - loaded_count
			if overflow_qty > 0:
				overflow_row = row.to_dict()
				overflow_row[day] = overflow_qty
				overflow_idx.append(overflow_row)

		overflow_df = DataFrame(overflow_idx) if overflow_idx else DataFrame()
		return loaded, overflow_df

	# ── push-out helper ───────────────────────────────────────────────────────

	def push_out(self, overflow_df: DataFrame, curr_day: str,
		next_day: str, shipment_pivot: DataFrame) -> DataFrame:
		"""
		Add overflow pallets to next day's demand in the pivot table.
		Mirrors push_out_shipment_plan_cluster from TLB.
		"""
		if overflow_df.empty or next_day is None:
			return shipment_pivot

		c = self.c
		key_cols = [c.TRUCK, c.ITEM, c.LOCATION]

		for _, row in overflow_df.iterrows():
			mask = True
			for k in key_cols:
				mask = mask & (shipment_pivot[k] == row[k])
			if shipment_pivot[mask].empty:
				continue
			shipment_pivot.loc[mask, next_day] += row.get(curr_day, 0)

		return shipment_pivot



#### ─────────────────────────────────────────────────────────────────────────────
6. POST-PROCESSOR
#### ─────────────────────────────────────────────────────────────────────────────


In [26]:

class LoadPlanPostProcessor:
	"""
	Aggregates loaded pallet lists into the four output DataFrames.
	"""

	def __init__(self, engine):
		self.engine = engine
		self.c = engine.c

	def build_load_plan_df(self) -> DataFrame:
		c = self.c
		if not self.engine.loaded_pallets_list:
			return DataFrame(columns=c.load_plan_columns)
		df = DataFrame(self.engine.loaded_pallets_list)
		df[c.out_planned_qty] = 1 # each row = 1 pallet
		# fill any missing output columns with 0
		for col in c.load_plan_columns:
			if col not in df.columns:
				df[col] = 0
		return df[c.load_plan_columns]

	def build_utilisation_df(self) -> DataFrame:
		c = self.c
		lp = self.build_load_plan_df()
		if lp.empty:
			return DataFrame(columns=c.utilisation_columns)

		rows = []
		for (day, truck), grp in lp.groupby([c.DAY, c.TRUCK]):
			eq = self.engine.equipment_lookup.get(truck, {})
			total_weight = grp[c.m_pallet_weight_kg].sum()
			total_area = grp[c.m_pallet_floor_area].sum()
			total_vol = grp[c.m_pallet_volume].sum()
			front_kg = grp[c.m_axle_front_load_kg].iloc[-1] if len(grp) else 0
			rear_kg = grp[c.m_axle_rear_load_kg].iloc[-1] if len(grp) else 0

			cap_w = eq.get(c.m_truck_max_payload_kg, 1)
			cap_a = eq.get(c.m_truck_floor_area_m2, 1)
			cap_v = eq.get(c.m_truck_volume_m3, 1)
			cap_f = eq.get(c.m_truck_max_front_axle_kg, 1)
			cap_r = eq.get(c.m_truck_max_rear_axle_kg, 1)

			rows.append({
				c.DAY: day,
				c.TRUCK: truck,
				c.out_weight_utilisation_pct: round(total_weight / cap_w * 100, 1),
				c.out_floor_area_utilisation_pct: round(total_area / cap_a * 100, 1),
				c.out_volume_utilisation_pct: round(total_vol / cap_v * 100, 1),
				c.out_front_axle_utilisation_pct: round(front_kg / cap_f * 100, 1),
				c.out_rear_axle_utilisation_pct: round(rear_kg / cap_r * 100, 1),
				c.out_axle_compliant: (
					front_kg <= eq.get(c.m_truck_max_front_axle_kg, float("inf"))
					and rear_kg <= eq.get(c.m_truck_max_rear_axle_kg, float("inf"))
				),
			})
		return DataFrame(rows, columns=c.utilisation_columns)

	def build_axle_weight_df(self) -> DataFrame:
		c = self.c
		lp = self.build_load_plan_df()
		if lp.empty:
			return DataFrame(columns=c.axle_weight_columns)
		# one row per pallet (sequence-level axle progression)
		lp[c.out_axle_compliant] = lp.apply(
			lambda row: self.engine.constraint_checker.axle_compliant(
				row[c.TRUCK],
				row[c.m_axle_front_load_kg],
				row[c.m_axle_rear_load_kg],
			), axis=1
		)
		return lp[c.axle_weight_columns].drop_duplicates()

	def build_exceptions_df(self) -> DataFrame:
		c = self.c
		if not self.engine.exception_list:
			return DataFrame(columns=c.exception_columns)
		df = DataFrame(self.engine.exception_list)
		for col in c.exception_columns:
			if col not in df.columns:
				df[col] = None
		return df[c.exception_columns]



#### ─────────────────────────────────────────────────────────────────────────────
7. LOAD PLAN MANAGER  —  orchestrator
#### ─────────────────────────────────────────────────────────────────────────────


In [27]:
class LoadPlanManager:
	"""
	Main entry point. Owns all state. Runs the day × truck loop.
	
	Usage:
	    manager = LoadPlanManager(
	        item_master_df=...,
	        lane_master_df=...,
	        load_equipment_metadata_df=...,
	        location_df=...,
	        shipment_plans_df=...,
	        sku_uom_df=...,
	        transport_asset_df=...,
	    )
	    (load_plan, utilisation, axle_weights, exceptions) = manager.run()
	"""

	def __init__(
		self,
		item_master_df: DataFrame,
		lane_master_df: DataFrame,
		load_equipment_metadata_df: DataFrame,
		location_df: DataFrame,
		shipment_plans_df: DataFrame,
		sku_uom_df: DataFrame,
		transport_asset_df: DataFrame,
	):
		# ── store raw inputs ─────────────────────────────────────────────────
		self.item_master_df = item_master_df
		self.lane_master_df = lane_master_df
		self.load_equipment_metadata_df = load_equipment_metadata_df
		self.location_df = location_df
		self.shipment_plans_df = shipment_plans_df
		self.sku_uom_df = sku_uom_df
		self.transport_asset_df = transport_asset_df

		# ── constants ────────────────────────────────────────────────────────
		self.c = LoadPlannerConstants()

		# ── sub-components ───────────────────────────────────────────────────
		self.preprocessor = LoadPlanPreprocessor(self)
		self.constraint_checker = ConstraintChecker(self)
		self.pallet_loader = PalletLoader(self)
		self.position_engine = PositionEngine(self)
		self.postprocessor = LoadPlanPostProcessor(self)

		# ── runtime state (populated by preprocessor) ─────────────────────
		self.no_input = False
		self.days: list = []
		self.shipment_pivot: DataFrame = DataFrame()
		self.equipment_df: DataFrame = DataFrame()
		self.equipment_lookup: dict = {}
		self.item_lookup: dict = {}
		self.lane_lookup: dict = {}

		# ── output accumulators ──────────────────────────────────────────────
		self.loaded_pallets_list: list = []
		self.exception_list: list = []

	# ── main solver ───────────────────────────────────────────────────────────

	def run(self) -> tuple:
		"""
		Entry point. Returns (load_plan_df, utilisation_df, axle_df, exceptions_df).
		"""
		c = self.c
		t0 = time.time()
		logger.info("LoadPlanManager: starting")

		# ── preprocess ───────────────────────────────────────────────────────
		self.preprocessor.run()

		if self.no_input:
			logger.error("LoadPlanManager: missing inputs, returning empty outputs")
			return (
				DataFrame(columns=c.load_plan_columns),
				DataFrame(columns=c.utilisation_columns),
				DataFrame(columns=c.axle_weight_columns),
				DataFrame(columns=c.exception_columns),
			)

		# ── day loop ─────────────────────────────────────────────────────────
		for i, day in enumerate(self.days):
			logger.info(f"LoadPlanManager: processing {day}")
			is_last_day = (i == len(self.days) - 1)
			next_day = self.days[i + 1] if not is_last_day else None

			# trucks available on this day
			available_trucks = self._get_available_trucks(day)
			if not available_trucks:
				logger.warning(f"  no trucks available on {day}")
				self._push_all_to_exception(day, "No truck available")
				continue

			# demand on this day
			day_demand = self.shipment_pivot[
				self.shipment_pivot[day] > 0
			].copy()

			if day_demand.empty:
				logger.info(f"  no demand on {day}")
				continue

			# ── truck loop ───────────────────────────────────────────────────
			for truck_id in available_trucks:
				truck_demand = day_demand[
					day_demand[c.TRUCK] == truck_id
				].copy()

				if truck_demand.empty:
					continue

				logger.info(f"  loading truck {truck_id}")

				loaded, overflow_df = self.pallet_loader.load_truck_for_day(
					truck_id, day, truck_demand
				)
				self.loaded_pallets_list.extend(loaded)

				# handle overflow
				if not overflow_df.empty:
					if not is_last_day:
						self.shipment_pivot = self.pallet_loader.push_out(
							overflow_df, day, next_day, self.shipment_pivot
						)
						logger.info(
							f"  pushed {len(overflow_df)} overflow rows to {next_day}"
						)
					else:
						for _, row in overflow_df.iterrows():
							self.exception_list.append({
								c.DAY: day,
								c.TRUCK: truck_id,
								c.ITEM: row.get(c.ITEM),
								c.LOCATION: row.get(c.LOCATION),
								c.out_exception_qty: row.get(day, 0),
								c.out_exception_reason: "Overflow on last day",
							})

			# zero out today's column — processed
			self.shipment_pivot[day] = 0

		# ── post-process ─────────────────────────────────────────────────────
		out_load_plan = self.postprocessor.build_load_plan_df()
		out_utilisation = self.postprocessor.build_utilisation_df()
		out_axle_weights = self.postprocessor.build_axle_weight_df()
		out_exceptions = self.postprocessor.build_exceptions_df()

		logger.info(f"LoadPlanManager: done in {time.time() - t0:.1f}s")
		return out_load_plan, out_utilisation, out_axle_weights, out_exceptions

	# ── private helpers ───────────────────────────────────────────────────────

	def _get_available_trucks(self, day: str) -> list:
		"""Return truck IDs that are available on `day`."""
		c = self.c
		if self.equipment_df.empty:
			return []

		if c.DAY in self.equipment_df.columns:
			avail = self.equipment_df[
				(self.equipment_df[c.DAY] == day)
				& (self.equipment_df[c.m_asset_available] > 0)
			][c.TRUCK].tolist()
		else:
			# no day-level availability — all trucks available every day
			avail = self.equipment_df[c.TRUCK].tolist()

		return avail

	def _push_all_to_exception(self, day: str, reason: str):
		"""Dump all demand for `day` into exception list."""
		c = self.c
		day_demand = self.shipment_pivot[self.shipment_pivot[day] > 0]
		for _, row in day_demand.iterrows():
			self.exception_list.append({
				c.DAY: day,
				c.TRUCK: row.get(c.TRUCK, ""),
				c.ITEM: row.get(c.ITEM, ""),
				c.LOCATION: row.get(c.LOCATION, ""),
				c.out_exception_qty: row[day],
				c.out_exception_reason: reason,
			})




#### ─────────────────────────────────────────────────────────────────────────────
USAGE EXAMPLE
#### ─────────────────────────────────────────────────────────────────────────────


In [33]:
item_master_df['is_stackable'] = item_master_df['stacking_limit']>0
item_master_df['is_fragile'] = False


In [36]:

"""
Replace these empty DataFrames with your actual data sources.
Column names must match the constants defined in LoadPlannerConstants.
"""

manager = LoadPlanManager(
	item_master_df = item_master_df,
	lane_master_df = lane_master_df,
	load_equipment_metadata_df = load_equipment_metadata_df,
	location_df = location_df,
	shipment_plans_df = shipment_plans_df,
	sku_uom_df = sku_uom_df,
	transport_asset_df = transport_asset_df,
)

load_plan, utilisation, axle_weights, exceptions = manager.run()
print("Load plan rows:    ", len(load_plan))
print("Utilisation rows:  ", len(utilisation))
print("Axle weight rows:  ", len(axle_weights))
print("Exception rows:    ", len(exceptions))


C:\Users\whysoseriousoni\AppData\Local\Temp\ipykernel_11468\787747988.py:45: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[c.m_item_weight_per_uom].fillna(0, inplace=True)
C:\Users\whysoseriousoni\AppData\Local\Temp\ipykernel_11468\787747988.py:46: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series thr

KeyError: 'units_per_pallet'

In [42]:
transport_asset_df

,transport_asset_id,asset_name,asset_type,axle_count,supports_refrigeration,supports_hazmat,max_weight_kg,assigned_from,assigned_to,current_status,created_at
0,1,TRUCK_CH_1DR,TRACTOR,4,False,False,40000.0,None,None,AVAILABLE,2026-05-19 10:37:06.157966


## Create Links for the following
1. Transport equipment assignment 🚚
   1. (transport_asset_df + load_equipment) [🛻+📦 record] 
2. 